## Exercise 1

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 12: Generative Adversarial Networks (GAN)
Niveau: Anfänger
Aufgabe 12.1: GAN Grundkonzept – Generator vs. Diskriminator

Lernziel: GAN-Prinzip (Spieltheorie) verstehen
Datensatz: Kein Datensatz (Konzepterklärung)
"""

import numpy as np
import matplotlib.pyplot as plt

print("=== GAN – Generative Adversarial Network ===")
print()
print("Zwei Netzwerke im Wettbewerb:")
print()
print("1. GENERATOR (G):")
print("   - Input:  Zufälliger Rauschvektor z (z ~ N(0,1))")
print("   - Output: Synthetisches Bild/Datenpunkt")
print("   - Ziel:   Den Diskriminator täuschen")
print()
print("2. DISKRIMINATOR (D):")
print("   - Input:  Bild (echt oder generiert)")
print("   - Output: P(echt) – Wahrscheinlichkeit [0,1]")
print("   - Ziel:   Echt von Gefälscht unterscheiden")
print()
print("Training-Gleichgewicht: Nash-Gleichgewicht")
print("Konvergenz: D kann nicht mehr unterscheiden (Ausgabe ≈ 0.5)")

# Simulierter GAN-Spielverlauf
epochen = 50
np.random.seed(42)
d_loss_real = np.exp(-np.linspace(0, 2, epochen)) * 0.5 + 0.1 + 0.05*np.random.randn(epochen)
d_loss_fake = np.exp(-np.linspace(0, 2, epochen)) * 0.5 + 0.1 + 0.05*np.random.randn(epochen)
g_loss = 1.0 - np.exp(-np.linspace(0, 2, epochen)) + 0.05*np.random.randn(epochen)

d_loss_real = np.clip(np.cumsum(d_loss_real)/epochen + np.random.randn(epochen)*0.1, 0.3, 0.8)[::-1]
d_loss_fake = np.clip(g_loss + np.random.randn(epochen)*0.1, 0.3, 0.8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].fill_between(range(epochen), 0, d_loss_real, alpha=0.3, color='#00E6FF')
axes[0].plot(d_loss_real, label='D(echte Daten)', color='#00E6FF', linewidth=2)
axes[0].fill_between(range(epochen), 0, d_loss_fake, alpha=0.3, color='#FF6B6B')
axes[0].plot(d_loss_fake, label='D(generierte Daten)', color='#FF6B6B', linewidth=2)
axes[0].axhline(0.5, color='gray', linestyle='--', label='Gleichgewicht (0.5)')
axes[0].set_title('Diskriminator-Ausgaben im Training')
axes[0].set_xlabel('Trainingsschritt')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1)

# GAN Architektur-Diagramm
ax = axes[1]
ax.axis('off')
for (x, y, label, farbe) in [
    (0.1, 0.5, 'Rauschen\nz~N(0,1)', '#7832FF'),
    (0.35, 0.5, 'Generator\nG(z)', '#0082FF'),
    (0.65, 0.7, 'Echte\nDaten', '#00E6FF'),
    (0.65, 0.5, 'Generierte\nBilder', '#0082FF'),
    (0.85, 0.6, 'Diskriminator\nD(x)', '#FF6B6B'),
]:
    ax.text(x, y, label, ha='center', va='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor=farbe, alpha=0.7))

for (x1, y1, x2, y2) in [(0.18, 0.5, 0.28, 0.5),
                           (0.42, 0.5, 0.57, 0.5),
                           (0.73, 0.7, 0.77, 0.65),
                           (0.73, 0.5, 0.77, 0.55)]:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='white', lw=2))
ax.set_facecolor('#1a1a2e')
ax.set_title('GAN Architektur')

plt.tight_layout()
plt.savefig('gan_konzept.png', dpi=150)
plt.show()

print("\n\nGAN Mini-Max Spieltheorie:")
print("G minimiert: log(1 - D(G(z)))")
print("D maximiert: log(D(x)) + log(1 - D(G(z)))")


## Exercise 2

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 12: Generative Adversarial Networks (GAN)
Niveau: Anfänger
Aufgabe 12.2: 1D GAN – Verteilungsanpassung

Lernziel: GAN lernt eine einfache Wahrscheinlichkeitsverteilung
Datensatz: Synthetische 1D Gaussverteilung
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)

ZIEL_MITTELWERT = 3.0
ZIEL_STD        = 0.5
LATENT_DIM      = 4
BATCH_SIZE      = 64

def echte_daten(n):
    return np.random.normal(ZIEL_MITTELWERT, ZIEL_STD, (n, 1)).astype('float32')

def rauschen(n):
    return np.random.randn(n, LATENT_DIM).astype('float32')

# Generator
generator = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(LATENT_DIM,)),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(1)
], name='Generator_1D')

# Diskriminator
diskriminator = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(1,)),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
], name='Diskriminator_1D')

d_optimizer = keras.optimizers.Adam(0.001)
g_optimizer = keras.optimizers.Adam(0.001)
bce = keras.losses.BinaryCrossentropy()

@tf.function
def train_schritt(real_data):
    z = tf.random.normal((BATCH_SIZE, LATENT_DIM))
    
    with tf.GradientTape() as d_tape:
        fake_data = generator(z, training=True)
        d_real = diskriminator(real_data, training=True)
        d_fake = diskriminator(fake_data, training=True)
        d_verlust = (bce(tf.ones_like(d_real), d_real) +
                     bce(tf.zeros_like(d_fake), d_fake))
    d_grads = d_tape.gradient(d_verlust, diskriminator.trainable_variables)
    d_optimizer.apply_gradients(zip(d_grads, diskriminator.trainable_variables))
    
    with tf.GradientTape() as g_tape:
        fake_data = generator(z, training=True)
        d_fake = diskriminator(fake_data, training=True)
        g_verlust = bce(tf.ones_like(d_fake), d_fake)
    g_grads = g_tape.gradient(g_verlust, generator.trainable_variables)
    g_optimizer.apply_gradients(zip(g_grads, generator.trainable_variables))
    
    return d_verlust, g_verlust

# Training
EPOCHEN = 500
d_losses, g_losses = [], []

for epoche in range(EPOCHEN):
    real = tf.constant(echte_daten(BATCH_SIZE))
    d_l, g_l = train_schritt(real)
    d_losses.append(float(d_l))
    g_losses.append(float(g_l))
    
    if epoche % 100 == 0:
        z = np.random.randn(1000, LATENT_DIM).astype('float32')
        fake = generator(z).numpy().ravel()
        print(f"Epoche {epoche:4d}: D={d_l:.3f} G={g_l:.3f} "
              f"| Generiert: μ={fake.mean():.2f} σ={fake.std():.2f} "
              f"| Ziel: μ={ZIEL_MITTELWERT} σ={ZIEL_STD}")

# Finale Visualisierung
z = np.random.randn(5000, LATENT_DIM).astype('float32')
generiert = generator(z).numpy().ravel()
echt = echte_daten(5000).ravel()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(echt, bins=50, density=True, alpha=0.6, label='Echte Daten', color='#00E6FF')
axes[0].hist(generiert, bins=50, density=True, alpha=0.6, label='Generiert', color='#FF6B6B')
axes[0].set_title('1D GAN: Verteilungsanpassung')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(d_losses, label='D Loss', color='#FF6B6B', alpha=0.7)
axes[1].plot(g_losses, label='G Loss', color='#00E6FF', alpha=0.7)
axes[1].set_title('Training Losses')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gan_1d_verteilung.png', dpi=150)
plt.show()


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 12: Generative Adversarial Networks (GAN)
Niveau: Anfänger
Aufgabe 12.3: DCGAN für MNIST – Einfacher Einstieg

Lernziel: Deep Convolutional GAN verstehen und implementieren
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, _), _ = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32')
X_train = (X_train - 127.5) / 127.5  # Normalisierung auf [-1, 1]

LATENT_DIM = 100
BATCH_SIZE  = 256

# Generator
def generator_modell():
    return keras.Sequential([
        keras.layers.Dense(7*7*256, use_bias=False, input_shape=(LATENT_DIM,)),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(),
        keras.layers.Reshape((7, 7, 256)),
        
        keras.layers.Conv2DTranspose(128, (5,5), strides=(1,1), padding='same', use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(),
        
        keras.layers.Conv2DTranspose(64, (5,5), strides=(2,2), padding='same', use_bias=False),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(),
        
        keras.layers.Conv2DTranspose(1, (5,5), strides=(2,2), padding='same',
                                      use_bias=False, activation='tanh'),
    ], name='Generator')

# Diskriminator
def diskriminator_modell():
    return keras.Sequential([
        keras.layers.Conv2D(64, (5,5), strides=(2,2), padding='same', input_shape=(28,28,1)),
        keras.layers.LeakyReLU(),
        keras.layers.Dropout(0.3),
        
        keras.layers.Conv2D(128, (5,5), strides=(2,2), padding='same'),
        keras.layers.LeakyReLU(),
        keras.layers.Dropout(0.3),
        
        keras.layers.Flatten(),
        keras.layers.Dense(1),
    ], name='Diskriminator')

generator     = generator_modell()
diskriminator = diskriminator_modell()

print("Generator:")
generator.summary()
print("\nDiskriminator:")
diskriminator.summary()

bce = keras.losses.BinaryCrossentropy(from_logits=True)
g_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
d_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)

@tf.function
def train_schritt(bilder):
    z = tf.random.normal((tf.shape(bilder)[0], LATENT_DIM))
    
    with tf.GradientTape() as d_tape, tf.GradientTape() as g_tape:
        fake = generator(z, training=True)
        d_real = diskriminator(bilder, training=True)
        d_fake = diskriminator(fake, training=True)
        
        d_loss = bce(tf.ones_like(d_real), d_real) + bce(tf.zeros_like(d_fake), d_fake)
        g_loss = bce(tf.ones_like(d_fake), d_fake)
    
    d_grads = d_tape.gradient(d_loss, diskriminator.trainable_variables)
    g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
    d_opt.apply_gradients(zip(d_grads, diskriminator.trainable_variables))
    g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))
    return d_loss, g_loss

# Training
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(60000).batch(BATCH_SIZE)
EPOCHEN = 5
fester_rausch = tf.random.normal((16, LATENT_DIM))

print("\nTraining DCGAN...")
for epoche in range(EPOCHEN):
    d_l_sum = g_l_sum = 0.0
    for batch in dataset:
        d_l, g_l = train_schritt(batch)
        d_l_sum += float(d_l)
        g_l_sum += float(g_l)
    
    n_batches = len(list(dataset))
    print(f"Epoche {epoche+1}/{EPOCHEN}: D={d_l_sum/n_batches:.4f} G={g_l_sum/n_batches:.4f}")

# Generierte Bilder
generierte = generator(fester_rausch, training=False)
generierte = (generierte + 1) / 2.0  # [-1,1] → [0,1]

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(generierte[i].numpy().squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('DCGAN Generierte MNIST Ziffern')
plt.tight_layout()
plt.savefig('dcgan_generiert.png', dpi=150)
plt.show()
